# Jana Eko Client - Comprehensive Test Suite

Tests the `jana-eko-client` library against the Jana Earth API (DEV environment).

**Pure client-side testing** - No AWS dependencies (SSM, boto3, etc.)

## Setup

```bash
# Install the client
pip install git+https://github.com/Jana-Earth-Data/jana-eko-client.git

# Set credentials (before launching Jupyter)
export JANA_API_URL="https://api-dev.jana.earth"
export JANA_USERNAME="dev-user"
export JANA_PASSWORD="your-password-here"

# Launch Jupyter
jupyter notebook
```

**Note**: Some endpoints may timeout due to materialized view recreation from recent ingestion work.

In [35]:
import os
import time
import warnings
from dataclasses import dataclass
from typing import Optional, Any

# Suppress Jupyter introspection warnings for async methods
warnings.filterwarnings('ignore', message='coroutine.*was never awaited')

# Import the jana-eko-client library
from eko_client import EkoUserClient
from eko_client.exceptions import (
    EkoClientError,
    EkoAuthenticationError,
    EkoAPIError,
    EkoRateLimitError,
    EkoNotFoundError,
)

# Configuration from environment
BASE_URL = os.environ.get("JANA_API_URL", "https://api-dev.jana.earth")
USERNAME = os.environ.get("JANA_USERNAME", "dev-user")
PASSWORD = os.environ.get("JANA_PASSWORD", "")

# Prompt for password if not in environment
if not PASSWORD:
    from getpass import getpass
    PASSWORD = getpass(f"Enter password for {USERNAME}: ")

print(f"Testing against: {BASE_URL}")
print(f"Username: {USERNAME}")
print(f"Password: {'*' * len(PASSWORD) if PASSWORD else '[NOT SET]'}")

Testing against: https://api-dev.jana.earth
Username: dev-user
Password: ************************


In [36]:
@dataclass
class TestResult:
    test_name: str
    method_called: str
    response_time_ms: float
    passed: bool
    error: Optional[str] = None
    record_count: Optional[int] = None

results: list[TestResult] = []

def test_method(name: str, method_name: str, func, *args, **kwargs) -> TestResult:
    """Test a client method and record results."""
    try:
        start = time.perf_counter()
        response = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        
        # Try to get record count
        record_count = None
        if isinstance(response, dict):
            if 'results' in response:
                record_count = len(response['results'])
            elif 'data' in response:
                record_count = len(response['data'])
        elif isinstance(response, list):
            record_count = len(response)
        
        result = TestResult(
            test_name=name,
            method_called=method_name,
            response_time_ms=round(elapsed, 1),
            passed=True,
            record_count=record_count
        )
        status = "✓ PASS"
    except Exception as e:
        elapsed = (time.perf_counter() - start) * 1000 if 'start' in locals() else 0
        result = TestResult(
            test_name=name,
            method_called=method_name,
            response_time_ms=round(elapsed, 1),
            passed=False,
            error=str(e)
        )
        status = "✗ FAIL"
    
    results.append(result)
    
    count_str = f" ({result.record_count} records)" if result.record_count is not None else ""
    print(f"  [{status}] {name:50} {result.response_time_ms:>8.1f}ms{count_str}")
    
    if not result.passed:
        print(f"         Error: {result.error}")
    
    return result

def section(title: str):
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")

print("Test framework loaded.")

Test framework loaded.


## Initialize Client & Authenticate

In [37]:
section("Client Initialization & Authentication")

try:
    start = time.perf_counter()
    
    # Initialize client with username/password
    client = EkoUserClient(
        base_url=BASE_URL,
        username=USERNAME,
        password=PASSWORD,
        timeout=60
    )
    
    elapsed = (time.perf_counter() - start) * 1000
    
    print(f"  [✓ PASS] Client initialized and authenticated          {elapsed:>8.1f}ms")
    results.append(TestResult(
        test_name="Client Authentication",
        method_called="EkoUserClient.__init__",
        response_time_ms=round(elapsed, 1),
        passed=True
    ))
    
    # Test get_user_info
    user_info = client.get_user_info()
    print(f"  [✓ INFO] Logged in as: {user_info.get('username', 'unknown')}")
    print(f"           Email: {user_info.get('email', 'N/A')}")
    
except EkoAuthenticationError as e:
    print(f"  [✗ FAIL] Authentication failed: {e}")
    raise
except Exception as e:
    print(f"  [✗ FAIL] Client initialization failed: {e}")
    raise


  Client Initialization & Authentication
  [✓ PASS] Client initialized and authenticated            1731.8ms
  [✓ INFO] Logged in as: dev-user
           Email: dev@jana.com


## Health Checks

## OpenAQ Endpoints

In [38]:
section("OpenAQ - Parameters & Locations")

test_method("OpenAQ Parameters", "get_openaq_parameters()", 
            client.get_openaq_parameters)

test_method("OpenAQ Locations", "get_openaq_locations(limit=5)", 
            client.get_openaq_locations, limit=5)

# Fix: country_codes must be a list
test_method("OpenAQ Locations (USA)", "get_openaq_locations(country_codes=['USA'])", 
            client.get_openaq_locations, country_codes=["USA"], limit=5)


  OpenAQ - Parameters & Locations
  [✓ PASS] OpenAQ Parameters                                    1339.1ms (30 records)
  [✓ PASS] OpenAQ Locations                                     1125.2ms (100 records)
  [✓ PASS] OpenAQ Locations (USA)                                386.9ms (100 records)


TestResult(test_name='OpenAQ Locations (USA)', method_called="get_openaq_locations(country_codes=['USA'])", response_time_ms=386.9, passed=True, error=None, record_count=100)

In [39]:
section("OpenAQ - Sensors & Measurements")

test_method("OpenAQ Sensors", "get_openaq_sensors(limit=5)", 
            client.get_openaq_sensors, limit=5)

test_method("OpenAQ Measurements", "get_openaq_measurements(limit=5)", 
            client.get_openaq_measurements, limit=5)

# Note: get_openaq_measurements doesn't filter by country, only by location_id or parameter
test_method("OpenAQ Measurements (PM2.5)", "get_openaq_measurements(parameter='pm25', limit=5)", 
            client.get_openaq_measurements, parameter="pm25", limit=5)


  OpenAQ - Sensors & Measurements
  [✓ PASS] OpenAQ Sensors                                        899.3ms (100 records)
  [✓ PASS] OpenAQ Measurements                                   323.0ms (0 records)
  [✓ PASS] OpenAQ Measurements (PM2.5)                           318.3ms (0 records)


TestResult(test_name='OpenAQ Measurements (PM2.5)', method_called="get_openaq_measurements(parameter='pm25', limit=5)", response_time_ms=318.3, passed=True, error=None, record_count=0)

In [40]:
section("OpenAQ - Statistics & Aggregations")

# Note: Some aggregations may timeout
test_method("OpenAQ Measurement Totals", "get_openaq_measurements_totals()", 
            client.get_openaq_measurements_totals)

test_method("OpenAQ Date Range", "get_openaq_measurements_date_range()", 
            client.get_openaq_measurements_date_range)

# These may timeout on dev
print("\n  Note: The following aggregations may timeout due to large dataset size:")
test_method("OpenAQ Country Totals", "get_openaq_measurements_country_totals()", 
            client.get_openaq_measurements_country_totals)

test_method("OpenAQ Parameter Totals", "get_openaq_measurements_parameter_totals()", 
            client.get_openaq_measurements_parameter_totals)


  OpenAQ - Statistics & Aggregations
  [✓ PASS] OpenAQ Measurement Totals                             318.4ms
  [✓ PASS] OpenAQ Date Range                                     316.3ms

  Note: The following aggregations may timeout due to large dataset size:
  [✓ PASS] OpenAQ Country Totals                                 330.6ms (176 records)
  [✓ PASS] OpenAQ Parameter Totals                               324.7ms (37 records)


TestResult(test_name='OpenAQ Parameter Totals', method_called='get_openaq_measurements_parameter_totals()', response_time_ms=324.7, passed=True, error=None, record_count=37)

## Climate TRACE Endpoints

In [41]:
section("Climate TRACE - Sectors & Countries")

test_method("Climate TRACE Sectors", "get_climatetrace_sectors()", 
            client.get_climatetrace_sectors)

test_method("Climate TRACE Countries", "get_climatetrace_countries()", 
            client.get_climatetrace_countries)

# Note: get_climatetrace_countries returns all countries, filter by country_code in emissions
test_method("Climate TRACE Countries (limited)", "get_climatetrace_countries(limit=10)", 
            client.get_climatetrace_countries, limit=10)


  Climate TRACE - Sectors & Countries
  [✓ PASS] Climate TRACE Sectors                                2356.3ms (26 records)
  [✓ PASS] Climate TRACE Countries                              1761.9ms (100 records)
  [✓ PASS] Climate TRACE Countries (limited)                    1798.6ms (100 records)


TestResult(test_name='Climate TRACE Countries (limited)', method_called='get_climatetrace_countries(limit=10)', response_time_ms=1798.6, passed=True, error=None, record_count=100)

In [42]:
section("Climate TRACE - Assets & Emissions")

test_method("Climate TRACE Assets", "get_climatetrace_assets(limit=5)", 
            client.get_climatetrace_assets, limit=5)

test_method("Climate TRACE Emissions", "get_climatetrace_emissions(limit=5)", 
            client.get_climatetrace_emissions, limit=5)

# Use asset filtering instead of direct country filtering
test_method("Climate TRACE Assets (USA)", "get_climatetrace_assets(country_code='US', limit=5)", 
            client.get_climatetrace_assets, country_code="US", limit=5)


  Climate TRACE - Assets & Emissions
  [✓ PASS] Climate TRACE Assets                                  877.5ms (5 records)
  [✓ PASS] Climate TRACE Emissions                               381.5ms (5 records)
  [✓ PASS] Climate TRACE Assets (USA)                            337.3ms (0 records)


TestResult(test_name='Climate TRACE Assets (USA)', method_called="get_climatetrace_assets(country_code='US', limit=5)", response_time_ms=337.3, passed=True, error=None, record_count=0)

In [43]:
section("Climate TRACE - Statistics")

test_method("Climate TRACE Emission Totals", "get_climatetrace_emissions_totals()", 
            client.get_climatetrace_emissions_totals)

test_method("Climate TRACE Sector Totals", "get_climatetrace_emissions_sector_totals()", 
            client.get_climatetrace_emissions_sector_totals)

test_method("Climate TRACE Country Totals", "get_climatetrace_emissions_country_totals()", 
            client.get_climatetrace_emissions_country_totals)


  Climate TRACE - Statistics
  [✓ PASS] Climate TRACE Emission Totals                         326.5ms
  [✗ FAIL] Climate TRACE Sector Totals                         60006.2ms
         Error: HTTP request failed: 
  [✓ PASS] Climate TRACE Country Totals                        56933.6ms (252 records)


TestResult(test_name='Climate TRACE Country Totals', method_called='get_climatetrace_emissions_country_totals()', response_time_ms=56933.6, passed=True, error=None, record_count=252)

## EDGAR Endpoints

In [44]:
section("EDGAR - Country Totals & Temporal Profiles")

test_method("EDGAR Country Totals", "get_edgar_country_totals(limit=5)", 
            client.get_edgar_country_totals, limit=5)

# Fix: parameter is country_code not country
test_method("EDGAR Country Totals (USA)", "get_edgar_country_totals(country_code='USA', limit=5)", 
            client.get_edgar_country_totals, country_code="USA", limit=5)

test_method("EDGAR Temporal Profiles", "get_edgar_temporal_profiles(limit=5)", 
            client.get_edgar_temporal_profiles, limit=5)


  EDGAR - Country Totals & Temporal Profiles


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR Country Totals                                  304.6ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR Country Totals (USA)                            359.8ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR Temporal Profiles                               343.3ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='EDGAR Temporal Profiles', method_called='get_edgar_temporal_profiles(limit=5)', response_time_ms=343.3, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

In [45]:
section("EDGAR - Grid Emissions")

# Grid emissions may timeout
test_method("EDGAR Grid Emissions", "get_edgar_grid_emissions(limit=5)", 
            client.get_edgar_grid_emissions, limit=5)

test_method("EDGAR Grid Emissions (2020)", "get_edgar_grid_emissions(year=2020, limit=5)", 
            client.get_edgar_grid_emissions, year=2020, limit=5)

test_method("EDGAR Grid Emissions (CO2)", "get_edgar_grid_emissions(gas='CO2', limit=5)", 
            client.get_edgar_grid_emissions, gas="CO2", limit=5)


  EDGAR - Grid Emissions


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR Grid Emissions                                  310.0ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR Grid Emissions (2020)                           314.4ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR Grid Emissions (CO2)                            301.3ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='EDGAR Grid Emissions (CO2)', method_called="get_edgar_grid_emissions(gas='CO2', limit=5)", response_time_ms=301.3, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

In [46]:
section("EDGAR - FastTrack")

test_method("EDGAR FastTrack", "get_edgar_fasttrack(limit=5)", 
            client.get_edgar_fasttrack, limit=5)


  EDGAR - FastTrack


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] EDGAR FastTrack                                       299.1ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='EDGAR FastTrack', method_called='get_edgar_fasttrack(limit=5)', response_time_ms=299.1, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

## Unified ESG Endpoints

In [47]:
section("ESG - Definitions & Metadata")

test_method("ESG Definitions", "get_definitions()", 
            client.get_definitions)

test_method("ESG Parameter Definitions", "get_parameter_definitions()", 
            client.get_parameter_definitions)

test_method("ESG Unit Definitions", "get_unit_definitions()", 
            client.get_unit_definitions)

test_method("ESG Source Definitions", "get_source_definitions()", 
            client.get_source_definitions)


  ESG - Definitions & Metadata


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Definitions                                       302.4ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Parameter Definitions                             306.6ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Unit Definitions                                  301.8ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Source Definitions                                302.7ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='ESG Source Definitions', method_called='get_source_definitions()', response_time_ms=302.7, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

In [48]:
section("ESG - Unified Data Access")

print("\n  Note: These cross-source queries may timeout on dev:")

test_method("ESG Unified Data (OpenAQ, USA)", "get_data(sources=['openaq'])", 
            client.get_data, sources=["openaq"], country_codes=["USA"], limit=5)

test_method("ESG Locations (OpenAQ, USA)", "get_locations(sources=['openaq'])", 
            client.get_locations, sources=["openaq"], country_codes=["USA"], limit=5)

test_method("ESG Sectors (ClimateTrace)", "get_sectors(sources=['climatetrace'])", 
            client.get_sectors, sources=["climatetrace"], limit=5)


  ESG - Unified Data Access

  Note: These cross-source queries may timeout on dev:


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Unified Data (OpenAQ, USA)                        342.3ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Locations (OpenAQ, USA)                           291.3ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Sectors (ClimateTrace)                            300.7ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='ESG Sectors (ClimateTrace)', method_called="get_sectors(sources=['climatetrace'])", response_time_ms=300.7, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

In [49]:
section("ESG - Quality Insights")

print("\n  Note: This endpoint may timeout due to materialized view recreation:")

test_method("ESG Quality Insights", "get_quality()", 
            client.get_quality)

test_method("ESG Quality (OpenAQ)", "get_quality(sources=['openaq'])", 
            client.get_quality, sources=["openaq"])


  ESG - Quality Insights

  Note: This endpoint may timeout due to materialized view recreation:


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Quality Insights                                  320.3ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Quality (OpenAQ)                                  306.7ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='ESG Quality (OpenAQ)', method_called="get_quality(sources=['openaq'])", response_time_ms=306.7, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

In [50]:
section("ESG - Correlations & Trends")

print("\n  Note: Cross-source correlations may timeout:")

test_method("ESG Correlations", "get_correlations()", 
            client.get_correlations, 
            sources=["openaq", "climatetrace"], 
            country_codes=["USA"])

test_method("ESG Trends", "get_trends()", 
            client.get_trends, 
            sources=["openaq"])


  ESG - Correlations & Trends

  Note: Cross-source correlations may timeout:
  [✗ FAIL] ESG Correlations                                        0.1ms
         Error: get_correlations_async() got an unexpected keyword argument 'country_codes'


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] ESG Trends                                            301.3ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



TestResult(test_name='ESG Trends', method_called='get_trends()', response_time_ms=301.3, passed=False, error='<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n</body>\r\n</html>\r\n', record_count=None)

## Export Functionality

In [51]:
section("ESG - Export Workflow")

print("\n  Testing async export creation and status check:")

try:
    # Create export
    export_result = test_method(
        "Create Export (CSV, OpenAQ, USA)", 
        "create_export()",
        client.create_export,
        format="csv",
        query={
            "sources": ["openaq"],
            "country_codes": ["USA"],
            "limit": 100
        }
    )
    
    if export_result.passed:
        # Note: In production, you'd poll for completion
        print("\n  Note: Export created successfully (async job queued)")
        print("        In production, poll get_export_status() until complete, then download_export()")
except Exception as e:
    print(f"  Export test skipped: {e}")


  ESG - Export Workflow

  Testing async export creation and status check:


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✗ FAIL] Create Export (CSV, OpenAQ, USA)                      306.8ms
         Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



## Error Handling Tests

In [52]:
section("Error Handling")

print("\n  Testing client exception handling:")

# Test 404 handling
try:
    client.get_openaq_locations(location_id=999999999)
    print("  [✗ FAIL] Should have raised EkoNotFoundError")
except EkoNotFoundError:
    print("  [✓ PASS] EkoNotFoundError raised correctly for invalid ID")
except Exception as e:
    print(f"  [? UNKNOWN] Unexpected exception: {type(e).__name__}: {e}")

# Test invalid authentication
try:
    bad_client = EkoUserClient(
        base_url=BASE_URL,
        username="invalid_user",
        password="wrong_password",
        timeout=10
    )
    print("  [✗ FAIL] Should have raised EkoAuthenticationError")
except EkoAuthenticationError:
    print("  [✓ PASS] EkoAuthenticationError raised correctly for bad credentials")
except Exception as e:
    print(f"  [? UNKNOWN] Unexpected exception: {type(e).__name__}: {e}")


  Error Handling

  Testing client exception handling:
  [? UNKNOWN] Unexpected exception: TypeError: get_openaq_locations_async() got an unexpected keyword argument 'location_id'


Error response is not valid JSON: Expecting value: line 1 column 1 (char 0). Status: 503


  [✓ PASS] EkoAuthenticationError raised correctly for bad credentials


## Test Summary

In [53]:
section("Test Summary")

total = len(results)
passed = sum(1 for r in results if r.passed)
failed = sum(1 for r in results if not r.passed)
timed = [r for r in results if r.response_time_ms > 0]
avg_time = sum(r.response_time_ms for r in timed) / max(1, len(timed))
max_time = max(timed, key=lambda r: r.response_time_ms, default=None)

print(f"\n  Total tests: {total}")
print(f"  Passed: {passed} ({100*passed/max(1,total):.1f}%)")
print(f"  Failed: {failed}")
print(f"  Average response time: {avg_time:.1f}ms")
if max_time:
    print(f"  Slowest test: {max_time.test_name} ({max_time.response_time_ms:.1f}ms)")

if failed > 0:
    print(f"\n  Failed tests:")
    for r in results:
        if not r.passed:
            print(f"    - {r.test_name}")
            print(f"      Method: {r.method_called}")
            print(f"      Error: {r.error}")

# Summary by category
print(f"\n  {'='*80}")
print(f"  Test Results by Category")
print(f"  {'='*80}")

categories = {}
for r in results:
    if 'OpenAQ' in r.test_name:
        cat = 'OpenAQ'
    elif 'Climate TRACE' in r.test_name or 'climatetrace' in r.test_name.lower():
        cat = 'Climate TRACE'
    elif 'EDGAR' in r.test_name:
        cat = 'EDGAR'
    elif 'ESG' in r.test_name:
        cat = 'ESG Unified'
    elif 'Export' in r.test_name:
        cat = 'Export'
    else:
        cat = 'Other'
    
    if cat not in categories:
        categories[cat] = {'total': 0, 'passed': 0, 'failed': 0}
    
    categories[cat]['total'] += 1
    if r.passed:
        categories[cat]['passed'] += 1
    else:
        categories[cat]['failed'] += 1

for cat, stats in sorted(categories.items()):
    pass_rate = 100 * stats['passed'] / max(1, stats['total'])
    print(f"  {cat:20} {stats['passed']:3}/{stats['total']:3} passed ({pass_rate:5.1f}%)")

print(f"\n  {'='*80}")
print(f"  jana-eko-client v0.2.0 - Test Suite Complete")
print(f"  {'='*80}")


  Test Summary

  Total tests: 39
  Passed: 19 (48.7%)
  Failed: 20
  Average response time: 3533.6ms
  Slowest test: Climate TRACE Sector Totals (60006.2ms)

  Failed tests:
    - Climate TRACE Sector Totals
      Method: get_climatetrace_emissions_sector_totals()
      Error: HTTP request failed: 
    - EDGAR Country Totals
      Method: get_edgar_country_totals(limit=5)
      Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>

    - EDGAR Country Totals (USA)
      Method: get_edgar_country_totals(country_code='USA', limit=5)
      Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>

    - EDGAR Temporal Profiles
      Method: get_edgar_temporal_profiles(limit=5)
      Error: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h